# Tetrahedral Field Accuracy Validation

The executable validation corpus has moved from `examples/tetra_field_accuracy_evaluation/` to `validation_test/tetra_field_accuracy_evaluation/`. This notebook is the result-saved docs layer: it reads the validation JSON sidecars, checks the accepted gates, and records runtime/debug metadata.

## Validation Lanes

- `analytical_reference_results.json`: uniform magnetization, tetra MSC vs hexa reference.
- `solver_comparison_results.json`: Radia tetra solver vs Radia hexa solver.
- `evaluation_results.json`: legacy NGSolve H-formulation extraction diagnostic, retained as a known issue.
- `ngsolve_reference_results.json`: independent A-formulation diagnostic; current local run records NaNs and is reported rather than asserted.
- `comparison_summary.json`: consolidated pass/known-issue status.

In [1]:
from pathlib import Path
import datetime as _dt
import hashlib
import importlib.metadata as _metadata
import json
import platform
import sys

ROOT = Path.cwd()
if not (ROOT / 'validation_test' / 'tetra_field_accuracy_evaluation').exists():
    ROOT = Path('S:/Radia/01_GitHub')
VALIDATION = ROOT / 'validation_test' / 'tetra_field_accuracy_evaluation'
DOCS = ROOT / 'docs' / 'tetra_field_accuracy_evaluation'
print(f'validation={VALIDATION.relative_to(ROOT)}')

validation=validation_test\tetra_field_accuracy_evaluation


In [2]:
summary = json.loads((VALIDATION / 'comparison_summary.json').read_text(encoding='utf-8'))
assert summary['overall_status'] == 'PASS_WITH_KNOWN_ISSUES', summary['overall_status']
assert summary['checks']['analytical_reference_pass']
assert summary['checks']['radia_solver_comparison_pass']
assert summary['checks']['legacy_ngsolve_extraction_known_issue_recorded']
print('overall=' + summary['overall_status'])
print('analytical_avg_percent={:.6g}'.format(summary['analytical_reference']['avg_percent']))
print('solver_avg_percent={:.6g}'.format(summary['radia_solver_comparison']['avg_percent']))
print('legacy_ngsolve_avg_percent={:.6g}'.format(summary['legacy_ngsolve_extraction']['avg_percent']))
print('ngsolve_finite_vectors={}/{}'.format(summary['ngsolve_reference']['finite_vectors'], summary['ngsolve_reference']['vectors']))

overall=PASS_WITH_KNOWN_ISSUES
analytical_avg_percent=1.49257e-12
solver_avg_percent=0.857542
legacy_ngsolve_avg_percent=295.634
ngsolve_finite_vectors=0/15


In [3]:
from IPython.display import Markdown, display

rows = [
    '| lane | status | avg error [%] | max error [%] |',
    '|---|---:|---:|---:|',
]
for key, label in [
    ('analytical_reference', 'tetra MSC vs hexa uniform M'),
    ('radia_solver_comparison', 'Radia tetra solver vs hexa solver'),
    ('legacy_ngsolve_extraction', 'legacy NGSolve extraction diagnostic'),
]:
    item = summary[key]
    status = item.get('status', 'pass')
    rows.append(f"| {label} | {status} | {item.get('avg_percent')} | {item.get('max_percent')} |")
display(Markdown('\n'.join(rows)))

| lane | status | avg error [%] | max error [%] |
|---|---:|---:|---:|
| tetra MSC vs hexa uniform M | pass | 1.49256869499686e-12 | 8.570446191805327e-12 |
| Radia tetra solver vs hexa solver | pass | 0.8575420391484876 | 2.9922326073443575 |
| legacy NGSolve extraction diagnostic | known_issue | 295.6335340647525 | 1579.470560283084 |

In [4]:

def _version(name):
    try:
        return _metadata.version(name)
    except Exception:
        return None

def _sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

json_files = sorted(VALIDATION.glob('*.json'))
py_files = sorted(VALIDATION.glob('*.py'))
result = {
    'schema': 'radia.docs.tetra_field_accuracy.validation.v1',
    'generated_at_utc': _dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'versions': {
        'python_version': sys.version.split()[0],
        'python_executable': sys.executable,
        'platform': platform.platform(),
        'radia_version': _version('radia'),
        'ngsolve_version': _version('ngsolve'),
        'numpy_version': _version('numpy'),
    },
    'validation_dir': 'validation_test/tetra_field_accuracy_evaluation',
    'summary': summary,
    'json_hashes': {p.name: {'sha256': _sha256(p), 'bytes': p.stat().st_size} for p in json_files},
    'source_hashes': {p.name: _sha256(p) for p in py_files},
    'checks': summary['checks'],
}
out = DOCS / 'tetra_field_accuracy_validation_results.json'
out.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f"wrote {out.relative_to(DOCS)}")
print(f"json_files={len(json_files)} source_files={len(py_files)}")


wrote tetra_field_accuracy_validation_results.json
json_files=5 source_files=5
